In [ ]:
import json
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

In [ ]:
labels_to_evaluate = [
    "research_type",
    "result_outcome",
    "affiliation",
    "problem_description",
    "goal_objective",
    "research_method", 
    "research_question",
    "hypothesis",
    "prediction",
    "contribution",
    "pseudocode",
    "open_source_code",
    "open_experiment_code",
    "train",
    "validation",
    "test",
    "results",
    "hardware_specification",
    "software_dependencies",
    "experiment_setup",
]

In [ ]:
# Ground truth of the evaluation of the conference papers
evaluations_file = "../../evaluations.json"

# Results of the LLM evaluation of the conference papers
llm_results_file = "../../SOTA/raw_data/run_01.json"
llm_results_file_base = llm_results_file.split("/")[-1].split(".")[0]

In [ ]:
# Read the ground truth evaluations
with open(evaluations_file, "r") as f:
    ground_truth_raw = json.load(f)

ground_truth = {}

for paper in ground_truth_raw:
    #print(paper)
    ground_truth[str(paper["index"])] = {}
    for label in labels_to_evaluate:
        if label in paper:
            ground_truth[str(paper["index"])][label] = {'result': paper[label]}
        else:
            ground_truth[str(paper["index"])][label] = {'result': None}
    
print(ground_truth)

In [ ]:
# Read the results of the LLM evaluation
with open(llm_results_file, "r") as f:
    llm_results = json.load(f)

print(llm_results)

In [ ]:
total = {"y_true": [], "y_pred": []}

results = {}

# Initialize results dictionary for each label
for label in labels_to_evaluate:

    results[label] = {"y_true": [], "y_pred": []}

    print(f"Evaluating label: {label}")

    for paper in llm_results:
        # Check if paper is a string of a paper id (skip metadata)
        if not paper.isdigit():
            continue
        
        if ground_truth[paper][label]["result"] == -1:
            # Skip papers that do not have a ground truth result for this label
            print(f"Skipping paper {paper} for label {label} due to missing ground truth result")
            continue
        
        if llm_results[paper][label]["result"] == -1:
            # Skip papers that do not have an LLM result for this label
            print(f"Skipping paper {paper} for label {label} due to missing LLM result")
            continue

        if label is "affiliation":
            # Combine collaboration and industry affiliations
            if ground_truth[paper]["affiliation"]["result"] == 2:
                ground_truth[paper]["affiliation"]["result"] = 1

            if llm_results[paper]["affiliation"]["result"] == 2:
                llm_results[paper]["affiliation"]["result"] = 1
            
        # Append the ground truth and LLM results to the total list
        total["y_true"].append(ground_truth[paper][label]["result"])
        total["y_pred"].append(llm_results[paper][label]["result"])
        
        # Add the ground truth and LLM results to the results dictionary
        results[label]["y_true"].append(ground_truth[paper][label]["result"])
        results[label]["y_pred"].append(llm_results[paper][label]["result"])

In [ ]:
for result in results:
    y_true = results[result]["y_true"]
    y_pred = results[result]["y_pred"]

    # Calculate accuracy
    accuracy = accuracy_score(y_true, y_pred)
    
    # Calculate F1 score 
    f1 = f1_score(y_true, y_pred)
    
    # Calculate confusion matrix
    conf_matrix = confusion_matrix(y_true, y_pred)

    print(f"Results for {result}:")
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print(f"F1 Score: {f1 * 100:.2f}%")
    print("Confusion Matrix:")
    print(conf_matrix)
    print("\n")
    #print(f"{result},{accuracy * 100:.2f},{f1 * 100:.2f},{len(y_pred)}")


In [ ]:
n_labels = len(labels_to_evaluate)
n_cols = 5
n_rows = (n_labels + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 3))
axes = axes.flatten()

for idx, label in enumerate(labels_to_evaluate):
    y_true = results[label]["y_true"]
    y_pred = results[label]["y_pred"]

    # Calculate accuracy
    accuracy = accuracy_score(y_true, y_pred)
    
    # Calculate F1 score
    f1 = f1_score(y_true, y_pred, average='macro')

    # Calculate confusion matrix
    cm = confusion_matrix(y_true, y_pred)

    ax = axes[idx]
    sns.heatmap(cm, annot=True, fmt="d", cmap="crest", ax=ax, cbar=False)
    ax.set_title(label)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

    # Add accuracy and F1 to the plot
    ax.text(
        0.5, -0.25,
        f"Accuracy: {accuracy*100:.1f}% | F1 Score: {f1*100:.2f}%",
        ha='center', va='top', transform=ax.transAxes, fontsize=10
    )

# Hide any unused subplots
for j in range(len(labels_to_evaluate), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.savefig(f"../../plots/SOTA/{llm_results_file_base}-heatmap.png", dpi=300)
plt.show()


In [ ]:
labels = labels_to_evaluate

tp_list, tn_list, fp_list, fn_list = [], [], [], []

for label in labels:
    y_true = results[label]["y_true"]
    y_pred = results[label]["y_pred"]
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    tp_list.append(tp)
    tn_list.append(tn)
    fp_list.append(fp)
    fn_list.append(fn)

x = np.arange(len(labels))
width = 0.2

plt.figure(figsize=(16, 6))
plt.bar(x - 1.5*width, tp_list, width, label='TP')
plt.bar(x - 0.5*width, tn_list, width, label='TN')
plt.bar(x + 0.5*width, fp_list, width, label='FP')
plt.bar(x + 1.5*width, fn_list, width, label='FN')
plt.xticks(
    x, 
    [f"{label} ({len(results[label]['y_true'])})" for label in labels], 
    rotation=45, ha='right'
)
plt.ylabel('Count')
plt.title('TP, TN, FP, FN for Each Label')
plt.legend()
plt.tight_layout()
plt.savefig(f"../../plots/SOTA/{llm_results_file_base}-cm-one.png", dpi=300)
plt.show()

In [ ]:
# Find the maximum number of samples for any label
max_samples = max(len(results[label]['y_true']) for label in labels)

# Plot TP, TN, FP, FN for each label in separate subplots
fig, axes = plt.subplots(5, 4, figsize=(20, 16))
axes = axes.flatten()

for idx, label in enumerate(labels):
    y_true = results[label]["y_true"]
    y_pred = results[label]["y_pred"]

    # Calculate accuracy
    accuracy = accuracy_score(y_true, y_pred)
    
    # Calculate F1 score
    f1 = f1_score(y_true, y_pred)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    counts = [tp, tn, fp, fn]
    n_samples = len(results[label]['y_true'])
    bars = axes[idx].bar(['TP', 'TN', 'FP', 'FN'], counts, color=['#4caf50', '#2196f3', '#f44336', '#ff9800'])
    axes[idx].set_title(label)
    axes[idx].set_ylabel(f'Count (Out of {n_samples})')
    axes[idx].set_ylim(0, n_samples)
    for bar in bars:
        height = bar.get_height()
        axes[idx].annotate(f'{height}',
                           xy=(bar.get_x() + bar.get_width() / 2, height),
                           xytext=(0, 3),
                           textcoords="offset points",
                           ha='center', va='bottom')
    # Print accuracy and F1 below the plot
    axes[idx].text(
        0.5, -0.25,
        f"Accuracy: {accuracy*100:.1f}% | F1 Score: {f1*100:.2f}%",
        ha='center', va='top', transform=axes[idx].transAxes, fontsize=10
    )
       
# Hide any unused subplots
for j in range(len(labels), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.savefig(f"../../plots/SOTA/{llm_results_file_base}-cm-many.png", dpi=300)
plt.show()